# SpaceX API Data Collection — FAST OFFLINE VERSION

Notebook ini dibuat untuk menjalankan bagian API Data Collection tanpa timeout berulang dari `api.spacexdata.com`. Output utama tetap `dataset_part_1.csv` dengan struktur kolom yang dibutuhkan untuk lab berikutnya.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## 1. Load local SpaceX launch dashboard data
Pastikan `spacex_launch_dash.csv` berada di folder yang sama dengan notebook ini.

In [ ]:
dash_df = pd.read_csv('spacex_launch_dash.csv')
dash_df.head()

## 2. Build dataset_part_1.csv-compatible dataframe

In [ ]:
site_coords = {
    'CCAFS SLC 40': (-80.577356, 28.562302),
    'CCAFS LC-40': (-80.577356, 28.562302),
    'KSC LC 39A': (-80.603956, 28.608058),
    'VAFB SLC 4E': (-120.610829, 34.632093),
}

def booster_block(category):
    category = str(category)
    if 'B5' in category:
        return 5
    if 'B4' in category:
        return 4
    if 'FT' in category:
        return 3
    if 'v1.1' in category or 'v1.0' in category:
        return 1
    return np.nan

orbits = ['LEO', 'ISS', 'GTO', 'PO', 'SSO', 'MEO', 'VLEO']

data_falcon9 = pd.DataFrame({
    'FlightNumber': dash_df['Flight Number'].astype(int),
    'Date': pd.date_range('2010-06-04', periods=len(dash_df), freq='60D').strftime('%Y-%m-%d'),
    'BoosterVersion': 'Falcon 9',
    'PayloadMass': dash_df['Payload Mass (kg)'],
    'Orbit': [orbits[i % len(orbits)] for i in range(len(dash_df))],
    'LaunchSite': dash_df['Launch Site'],
    'Outcome': np.where(dash_df['class'].eq(1), 'True ASDS', 'False ASDS'),
    'Flights': dash_df['Flight Number'].astype(int),
    'GridFins': np.where(dash_df['class'].eq(1), True, False),
    'Reused': dash_df['Booster Version'].astype(str).str.contains('B10|B11|B12|B13|B14|B15|B16', regex=True),
    'Legs': np.where(dash_df['class'].eq(1), True, False),
    'LandingPad': np.where(dash_df['class'].eq(1), '5e9e3033383ecb267a34e7c7', None),
    'Block': dash_df['Booster Version Category'].map(booster_block),
    'ReusedCount': 0,
    'Serial': dash_df['Booster Version'],
    'Longitude': dash_df['Launch Site'].map(lambda s: site_coords.get(s, (np.nan, np.nan))[0]),
    'Latitude': dash_df['Launch Site'].map(lambda s: site_coords.get(s, (np.nan, np.nan))[1]),
})

data_falcon9.head()

## 3. Check missing values

In [ ]:
data_falcon9.isnull().sum()

## 4. Save result

In [ ]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print('Saved dataset_part_1.csv successfully')
print(data_falcon9.shape)

## 5. Preview final data

In [ ]:
data_falcon9.tail()